In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

# 1. 基础GPU可用性检查
print("="*50)
print("PyTorch GPU 可用性验证示例")
print("="*50)
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"当前设备: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 数量: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")
    print("警告: 将使用 CPU 进行训练，GPU 加速未启用。")
print("-"*50)

# 2. 定义一个简单的测试用神经网络
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 320)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 3. 准备数据 (使用MNIST的极小子集进行快速验证)
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])
# 使用很小的数据集，仅用于验证
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)

# 4. 初始化模型、损失函数和优化器，并将它们移动到指定设备
model = SimpleCNN().to(device)  # 关键步骤：将模型移到GPU
criterion = nn.CrossEntropyLoss().to(device)  # 损失函数也可明确设备
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# 5. 进行一个简短的训练循环，观察GPU使用情况
print("开始简短训练循环以验证GPU计算...")
model.train()
start_time = time.time()

for epoch in range(3):  # 只跑3个epoch用于验证
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        if batch_idx >= 10:  # 每个epoch只跑10个batch，加快验证速度
            break

        # 关键步骤：将数据移动到与模型相同的设备
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'  训练轮次 [{epoch+1}/3], 平均损失: {running_loss/10:.4f}')

end_time = time.time()
print(f"训练耗时: {end_time - start_time:.2f} 秒")
print("-"*50)

# 6. 进行一个简单的推理测试
print("进行GPU推理测试...")
model.eval()
with torch.no_grad():
    test_data, _ = next(iter(train_loader))
    test_data = test_data.to(device)
    output = model(test_data[:5])  # 只预测前5个样本
    _, predicted = torch.max(output.data, 1)
    print(f"  前5个样本的预测标签: {predicted.cpu().numpy()}")  # 将结果移回CPU用于显示
print("-"*50)

# 7. 显存使用情况检查（如果可用）
if torch.cuda.is_available():
    print("GPU 显存使用情况总结:")
    print(f"  当前显存分配: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB")
    print(f"  最大显存分配: {torch.cuda.max_memory_allocated(0)/1024**2:.2f} MB")
    print(f"  当前显存缓存: {torch.cuda.memory_reserved(0)/1024**2:.2f} MB")

# 8. 最终验证
print("="*50)
if torch.cuda.is_available() and torch.cuda.memory_allocated(0) > 0:
    print("✅ 验证通过！PyTorch GPU 正在正常工作。")
    print("   模型训练和推理已成功在GPU上执行。")
else:
    print("⚠️  注意：未检测到GPU活动。")
    print("   请检查CUDA和PyTorch-GPU版本的兼容性，或显卡驱动。")
print("="*50)

PyTorch GPU 可用性验证示例
PyTorch 版本: 2.5.1+cu121
CUDA 是否可用: True
当前设备: NVIDIA GeForce RTX 3080
CUDA 版本: 12.1
GPU 数量: 1
--------------------------------------------------
Failed to download (trying next):
HTTP Error 404: Not Found



31.1%